# 15 · 분류 모델 구현과 평가 — 타이타닉 생존자 분류
교과서 **Ⅱ-05(108~112쪽)** 의 분류 모델 구현과 평가를 **다른 사례**로 해 봅니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

**문제 정의** — 1912년 타이타닉호 승객의 객실 등급 · 성별 · 나이 · 동승 가족 · 요금으로 **생존(1) / 사망(0)** 을 분류한다.
정답이 두 갈래 → **이진 분류** → 지도학습.

> ⚠️ 실제 사고로 희생된 사람들의 기록입니다. 숫자 뒤에 사람이 있다는 것을 기억하고, «누가 살아남기 쉬웠는가»가 **당시 사회의 불평등**(객실 등급, «여성과 아이 먼저»)을 보여 준다는 점도 함께 생각해 봅니다(Ⅲ단원 윤리와 연결).

| 단계 | 핵심 함수 |
|---|---|
| ① 탐색 | `load_dataset` `value_counts` `groupby().mean()` |
| ② 전처리 | `fillna` `map` `get_dummies` |
| ③ 분리 | `train_test_split(..., stratify=y)` |
| ④ 학습 | `LogisticRegression` `DecisionTreeClassifier` `KNeighborsClassifier` |
| ⑤ 예측 | `predict` `predict_proba` |
| ⑥ 평가 | `accuracy_score` `confusion_matrix` `classification_report` |
| ⑦ 해석 | 기준선 · 정밀도/재현율 · 임계값 · 특성 해석 |

In [ ]:
# (그래프에 한글이 네모로 나오면 한 번만 실행)
!pip -q install koreanize-matplotlib
import koreanize_matplotlib

## ① 불러오기와 탐색

In [ ]:
import pandas as pd, numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic')
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
print(df.shape)
df.head()

| 열 | 뜻 |
|---|---|
| survived | **정답** 1 생존 · 0 사망 |
| pclass | 객실 등급 1 · 2 · 3 |
| sex | 성별 |
| age | 나이 |
| sibsp · parch | 함께 탄 형제·배우자 수 · 부모·자녀 수 |
| fare | 요금 |
| embarked | 탑승 항구 C · Q · S |

💡 생존자 비율 → `value_counts(normalize=True)`

In [ ]:
print(df['survived'].____(normalize=True).round(3))
print(df.groupby('sex')['survived'].mean().round(3))
print(df.groupby('pclass')['survived'].mean().round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=df, x='sex', y='survived', ax=ax[0])
sns.barplot(data=df, x='pclass', y='survived', ax=ax[1])
sns.histplot(data=df, x='age', hue='survived', bins=30, ax=ax[2])
plt.show()

**읽기** — 전체 생존율 약 38%. 성별 · 객실 등급에 따라 크게 다릅니다. → 이 둘이 **핵심 속성**일 가능성이 큽니다.

**기준선(baseline)** — 아무것도 안 배우고 «**전부 사망**»이라고만 찍어도 정확도가 약 **62%**! 우리 모델은 이것보다 확실히 나아야 의미가 있습니다.

---
## ② 전처리 — 결측치 · 범주를 숫자로
- `age` 결측 → **중앙값**으로 대체(연속형) · `embarked` 결측 → **최빈값**(범주형) — Ⅱ-02 표 Ⅱ-5
- 모델은 글자를 못 읽으니 `sex` → 0/1 로 (`map`)
- `embarked` 처럼 순서 없는 범주는 열 여러 개(0/1)로 펼침 (`get_dummies`, 원-핫 인코딩)

In [ ]:
print(df.isnull().sum())
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['sex'] = df['sex'].map({'male': 0, 'female': ____})
df = pd.get_dummies(df, columns=['embarked'], dtype=int)
df.head()

---
## ③ 데이터 분리 — `stratify=y`
`stratify=y` 를 주면 훈련 · 테스트 양쪽의 **생존 비율이 같게** 나뉩니다(불균형 데이터에서 중요).

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=____)
print(X_train.shape, X_test.shape, '훈련 생존율', y_train.mean().round(3), '테스트 생존율', y_test.mean().round(3))

---
## ④ 학습 — 세 모델을 같은 조건으로
- **로지스틱 회귀** — 점수를 S자 곡선에 넣어 **생존 확률**로 (Ⅱ-04 농구 슛 예제와 같은 원리)
- **의사결정트리** — 예/아니요 질문으로 나눔 (`max_depth=3` 으로 깊이 제한)
- **k-최근접 이웃** — 가까운 5명의 다수결. **거리 기반이라 단위를 맞추는 `StandardScaler`** 를 앞에 붙임

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

log = LogisticRegression(max_iter=1000).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)

for 이름, m in [('로지스틱', log), ('트리', tree), ('kNN', knn)]:
    print(f'{이름:5s} 훈련 {m.score(X_train, y_train):.3f}  테스트 {m.score(X_test, y_test):.3f}')
print('기준선(전부 사망)', round(1 - y_test.mean(), 3))

In [ ]:
plt.figure(figsize=(16, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=['died', 'survived'], filled=True, fontsize=8)
plt.show()

**트리 읽기** — 맨 위 질문(가장 먼저 나누는 속성)이 무엇인가요? `samples` 는 그 칸에 들어온 승객 수, `value` 는 [사망, 생존] 수입니다.

---
## ⑤ 예측 — `predict` 와 `predict_proba`
- `predict` → 0 또는 1
- `predict_proba` → **[사망 확률, 생존 확률]**. 생존 확률이 0.5 를 넘으면 1 로 분류합니다.

In [ ]:
prob = log.predict_proba(X_test)[:, 1]
pred = log.predict(X_test)
pd.DataFrame({'실제': y_test.values[:10], '예측': pred[:10], '생존 확률': prob[:10].round(2)})

---
## ⑥ 평가 — 정확도 · 혼동 행렬 · 정밀도 · 재현율 (교과서 108~112쪽)
**혼동 행렬(오분류표)**
|  | 예측 사망(0) | 예측 생존(1) |
|---|---|---|
| **실제 사망(0)** | 맞힘(TN) | 헛경보(FP) |
| **실제 생존(1)** | 놓침(FN) | 맞힘(TP) |

- **정확도** = 맞힌 수 ÷ 전체
- **정밀도** = «생존이라고 한 것» 중 진짜 생존 = TP ÷ (TP + FP)
- **재현율** = 진짜 생존자 중 찾아낸 비율 = TP ÷ (TP + FN)
- **F1** = 정밀도와 재현율의 조화 평균
💡 혼동 행렬 함수 → `confusion_matrix(실제, 예측)`

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
print('정확도 :', round(accuracy_score(y_test, pred), 3))
cm = ____(y_test, pred)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=['died', 'survived']).plot(cmap='Blues')
plt.show()
print(classification_report(y_test, pred, target_names=['died', 'survived']))

**classification_report 읽기** — `precision` 정밀도 · `recall` 재현율 · `f1-score` · `support` 그 집단의 실제 개수. `accuracy` 줄이 정확도.

### 임계값(threshold) 바꿔 보기
«생존 확률 0.5 이상이면 생존»이라는 기준을 **0.3** 으로 낮추면? → 생존이라고 하는 경우가 늘어 **재현율 ↑ · 정밀도 ↓**. 구조 대상을 놓치면 안 되는 상황이라면 어느 쪽이 나을까요?

In [ ]:
from sklearn.metrics import precision_score, recall_score
for th in [0.3, 0.5, 0.7]:
    p = (prob >= th).astype(int)
    print(f'임계값 {th}  정확도 {accuracy_score(y_test, p):.3f}  정밀도 {precision_score(y_test, p):.3f}  재현율 {recall_score(y_test, p):.3f}')

### 어떤 속성이 중요했나 — 로지스틱 계수와 트리 중요도
- 로지스틱 계수 **+** 는 생존 확률을 높이는 쪽, **−** 는 낮추는 쪽
- 트리 `feature_importances_` 는 나누는 데 얼마나 많이 쓰였는가

In [ ]:
pd.DataFrame({'로지스틱 계수': log.coef_[0].round(3), '트리 중요도': tree.feature_importances_.round(3)}, index=X.columns).sort_values('트리 중요도', ascending=False)

### 새 승객 예측
💡 열 순서와 이름이 학습 때와 같아야 합니다(`X.columns`).

In [ ]:
새 = pd.DataFrame([[3, 0, 25, 0, 0, 7.9, 0, 0, 1],      # 3등실 · 남 · 25세 · 혼자 · S항
                   [1, 1, 30, 1, 0, 80.0, 1, 0, 0]],   # 1등실 · 여 · 30세 · 배우자와 · C항
                  columns=X.columns)
print(log.predict_proba(새)[:, 1].round(2))

---
## ⑦ 결과 해석 — 문장으로
> 로지스틱 회귀 모델은 테스트 승객 ___명에서 정확도 ___로, 기준선(전부 사망 ___)보다 ___ 높다.
> 생존자 재현율은 ___ 로, 실제 생존자 ___명 중 ___명을 놓쳤다. 임계값을 0.3 으로 낮추면 재현율이 ___ 까지 오르지만 정밀도는 ___ 로 떨어진다.
> 가장 중요한 속성은 ___ 와 ___ 였다. 이것은 «모델이 공정하다»는 뜻이 아니라, **당시 구조 과정의 불평등이 데이터에 그대로 담겼다**는 뜻이다 — 이런 데이터로 오늘날 사람을 판정하면 안 된다(데이터 편향, Ⅱ-01 · Ⅲ-04).

---
### 정답 예시 (막혔을 때만)
- ① `value_counts` · ② `1` · ③ `stratify=y` · ⑥ `confusion_matrix`
- 세 모델 모두 테스트 정확도 약 0.78~0.82 — 기준선 0.62 보다 확실히 높음